In [ ]:
# Import necessary packages
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import re
import string
import warnings
warnings.filterwarnings('ignore')

# NLTK data
import nltk

# Model
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report, accuracy_score

# Semtiment analysis
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
# Import pipeline function
from transformers import pipeline


In [ ]:
# Download required NLTK data
nltk.download('punkt')  # Sentence tokenizer
nltk.download('words')  # English word list
nltk.download('stopwords')  # Common stopwords
nltk.download('wordnet')  # WordNet lexical database
nltk.download('punkt_tab') # Download punkt_tab resource

# Import WordNet, a lexical database used for lemmatization and semantic analysis
from nltk.corpus import wordnet
# Import a list of common English stopwords (e.g., "the", "and", "is") used for filtering out non-informative words
from nltk.corpus import stopwords


## Load and inspect the data

In [ ]:
bank_df = pd.read_csv('complaints_banking_2023.csv')
bank_df.head()


In [ ]:
# Check for missing values
bank_df.isnull().sum()
# Check data types
bank_df.dtypes

In [ ]:
# Convert the complaint date column to datetime (only date)

bank_df['Date Received'] = pd.to_datetime(bank_df['Date Received']).dt.date

# Get earliest and latest complaint
earliest_date = bank_df['Date Received'].min()
latest_date = bank_df['Date Received'].max()

print("Earliest Complaint Date:", earliest_date)
print("Latest Complaint Date:", latest_date)

# Text Preprocessing


In [ ]:
# Get list of stopwords
stp_wrds_eng = stopwords.words('english')

def clean_my_text(df_text_col):
  # Lower case
  clean_text = df_text_col.apply(lambda x: x.lower())
  # Remove numbers
  clean_text = clean_text.apply(lambda x: re.sub(r'\d+', '', x))
  # Remove punctuation
  clean_text = clean_text.apply(lambda x:re.sub(f'[{re.escape(string.punctuation)}]', '', x))
  # Tokenize reviews
  clean_text = clean_text.apply(lambda x: nltk.word_tokenize(x))
  # Remove stopwords
  clean_text = clean_text.apply(lambda x: [word for word in x if word not in stp_wrds_eng])
  # Lemmatize
  lemmatizer = nltk.stem.WordNetLemmatizer()
  clean_text = clean_text.apply(lambda x: [lemmatizer.lemmatize(word) for word in x])
  return clean_text



In [ ]:
clean_text = clean_my_text(bank_df['Complaint Description'])

bank_df['clean_text'] = clean_text
bank_df.head()


## Text Feature Engineering

In [ ]:
# Create TF-IDF features
tfidf = TfidfVectorizer(max_features=5000)
# Convert the cleaned text into a TF-IDF matrix
X_tfidf = tfidf.fit_transform(bank_df['clean_text'].apply(lambda x: ' '.join(x)))


## Complaint Classification

In [ ]:
from sklearn.metrics import confusion_matrix

# Define the target variable (department)
y = bank_df['Department']
# Build a classification model with input = x_tdidf and output = y
X_train, X_test, y_train, y_test = train_test_split(X_tfidf, y, test_size=0.2, random_state=42)

# Naive Bayes Model
nb = MultinomialNB()
nb.fit(X_train, y_train)
nb_preds = nb.predict(X_test)

print("Naive Bayes Performance:")
print("Accuracy:", accuracy_score(y_test, nb_preds))
print(classification_report(y_test, nb_preds))

# confusion matrix
cm = confusion_matrix(y_test, nb_preds)
# plot confusion matrix
sns.heatmap(cm, annot=True, fmt='d')
plt.xlabel('Predicted')
plt.ylabel('True')
plt.show()

# Logistic Regression Model
logreg = LogisticRegression(max_iter=1000)
logreg.fit(X_train, y_train)
logreg_preds = logreg.predict(X_test)

print("Logistic Regression Performance:")
print("Accuracy:", accuracy_score(y_test, logreg_preds))
print(classification_report(y_test, logreg_preds))

cm = confusion_matrix(y_test, logreg_preds)
# plot confusion matrix
sns.heatmap(cm, annot=True, fmt='d')
plt.xlabel('Predicted')
plt.ylabel('True')
plt.show()


##Apply VADER
VADER (Valence Aware Dictionary and sEntiment Reasoner) is a lexicon and rule-based, open-source sentiment analysis tool specifically designed for social media text and online reviews. It efficiently classifies text as positive, negative, or neutral based on word intensity, punctuation (e.g., "!!!"), capitalization, and intensifiers (e.g., "very").

In [ ]:
# Apply VADER

sia = SentimentIntensityAnalyzer()
# Predict vader score and apply to the data
bank_df['vader_scores'] = bank_df['Complaint Description'].apply(
    lambda x: sia.polarity_scores(str(x))
)

# Extract VADER compound score
bank_df['vader_compound'] = bank_df['vader_scores'].apply(
    lambda x: x['compound']
)
# Convert score to sentiment
def vader_label(score):
    if score >= 0.05:
        return 'positive'
    elif score <= -0.05:
        return 'negative'
    else:
        return 'neutral'

bank_df['vader_sentiment'] = bank_df['vader_compound'].apply(vader_label)



In [ ]:
bank_df.head()

In [ ]:
# Average VADER compound score by department

dept_summary = (
    bank_df.groupby('Department')['vader_compound']
      .agg(mean='mean', median='median', n='count')
      .sort_values('mean')
)

plt.figure()
plt.bar(dept_summary.index, dept_summary['mean'])
plt.xticks(rotation=45, ha='right')
plt.xlabel('Department')
plt.ylabel('Average VADER compound score')
plt.title('Average Complaint Sentiment by Department (VADER compound)')
plt.tight_layout()
plt.show()

# (Optional) print the table you’re plotting (nice for the report)
dept_summary


In [ ]:

# Optional KDE (no seaborn) — smooth density curve
plt.figure()
bank_df['vader_compound'].plot(kind='kde')
plt.xlabel('VADER compound score')
plt.ylabel('Density')
plt.title('KDE of Complaint Sentiment (VADER compound)')
plt.tight_layout()
plt.show()


## Apply BERT and discuss model performance

In [ ]:
# Load sentimet analysis with BERT
classifier = pipeline(
    "sentiment-analysis",
    model = "distilbert-base-uncased-finetuned-sst-2-english"
    )
# Run analysis and collect results

labels = []
scores = []

for complaint in bank_df['Complaint Description']:
    # Return the prediction dictionary using [0]
    # Add truncation=True to handle sequences longer than the model's max length
    out = classifier(complaint, truncation=True)[0]
    labels.append(out['label'])
    scores.append(out['score'])

bank_df['bert_sentiment'] = labels
bank_df['bert_confidence'] = scores


# Run sentiment analysis and collect results for bank_df
labels = []
scores = []

# Change sentiment to 'neutral' if confidence is <0.6
def map_to_three_classes(label, score, threshold=0.6):
    if score < threshold:
        return "NEUTRAL"
    return label

bank_df['sentiment_3class'] = [
    map_to_three_classes(l, s)
    for l, s in zip(bank_df['bert_sentiment'], bank_df['bert_confidence'])
]

bank_df

# Summary Report:
### Project overview:
The complaints dataset was loaded into Python, data types were checked and corrected, and the overall date range was identified to confirm coverage of the analysis period. Complaint descriptions were preprocessed using a standardized NLP pipeline (lowercasing, removing numbers and punctuation, stopword removal, and lemmatization). Cleaned complaint text was then transformed into a TF-IDF feture matrix for downstream modeling.

### Department classification
To automate triaging, the “Department” field was used as the target variable in a multi-class text classification task. Two baseline models were trained and evaluated: Multinomial Naive Bayes and Logistic Regression. Naive Bayes achieved ~0.685 accuracy, but performed poorly on minority departments (e.g., Others and Remittance), indicating sensitivity to class imbalance. Logistic Regression improved overall performance to ~0.750 accuracy and increased macro-level performance, showing stronger and more consistent discrimination across the major departments (e.g., Mortgage, Loans, Credit Reports), though small classes remained more difficult.

### Sentiment analysis and insights (VADER).
SentimentIntensityAnalyzer (VADER) was applied to complaint text to compute positive/neutral/negative proportions and a compound score between −1 and +1, representing overall sentiment intensity. Two plots were generated to support business interpretation:

1. Average VADER compound score by department, highlighting which product areas are associated with the most negative customer sentiment and therefore may require targeted process or product improvements.
2. Distribution of compound scores, showing the overall severity spread of complaints and enabling threshold-based escalation (e.g., very negative scores as “critical” complaints).

### Transformer-based sentiment analysis (BERT)
In addition to VADER, a pre-trained DistilBERT sentiment model (fine-tuned on SST-2) was applied to the complaint descriptions. BERT produced highly confident sentiment predictions, with the majority of complaints classified as NEGATIVE with confidence scores close to 1.0. This outcome is consistent with the nature of the dataset, which consists primarily of grievances rather than neutral customer feedback.
To extend BERT’s binary output to a three-class setting, a confidence-based heuristic was applied: predictions with confidence below a predefined threshold were labeled as NEUTRAL. This allowed limited alignment with the three-class sentiment framing used elsewhere in the analysis.

### Comparison and interpretation of VADER vs BERT
While both approaches identify widespread negative sentiment, they capture different aspects of customer dissatisfaction:
VADER provides a graded measure of sentiment intensity, distinguishing mild dissatisfaction from extreme frustration.
BERT produces highly confident categorical judgments, often labeling complaints as negative even when VADER indicates positive or neutral sentiment at the lexical level.
Observed disagreements (e.g., VADER-positive but BERT-negative cases) typically correspond to complaints that are factually resolved but still embedded in a grievance context. This highlights that BERT captures contextual complaint framing, whereas VADER emphasizes lexical sentiment inten

### Results interpretation: Combining both sentiment approaches offers complementary value:
VADER compound scores can be used to prioritize complaints by severity and urgency.
BERT sentiment labels can serve as a robust signal for detecting grievance framing at scale.
Together with department classification, these methods enable automated routing, severity-based prioritization, and department-level monitoring of customer dissatisfaction.

### Business use
This project demonstrates a complete NLP-driven grievance analysis pipeline, integrating text preprocessing, TF-IDF–based classification, lexicon-based sentiment analysis, and transformer-based sentiment inference. The combined use of VADER and BERT provides both interpretable severity scores and robust contextual sentiment detection, enabling actionable insights to improve complaint handling efficiency, customer experience, and regulatory risk monitoring.